# Ablation Study — AugCRNN-T

Bu notebook makalenin iki ablation tablosunu üretir. **Hepsi aynı ortamda (Kaggle T4) ölçülür**, böylece sayılar birbiriyle ve 84.54 ile tutarlı olur.

## Tablo A — Augmentation ablation (3 eğitim, ~6 saat)

| Konfigürasyon | Wide photo | Elastic | Morph |
|---|:---:|:---:|:---:|
| CRNN-L (mevcut, 78.06) | ✗ | ✗ | ✗ |
| `--aug-mode photo` | ✓ | ✗ | ✗ |
| `--aug-mode elastic` | ✓ | ✓ | ✗ |
| `--aug-mode morph` | ✓ | ✗ | ✓ |
| AugCRNN-T (mevcut, 84.54) | ✓ | ✓ | ✓ |

## Tablo B — Lexicon/trigram ablation (eğitim yok, ~15 dk)

Tek modelin çıktısına dört farklı post-processing uygulanır.

## Gerekli Input (Add Data)
1. IAM word dataset (`words.txt` + `words/`)
2. `berhat-v3-augmented-model` — Tablo B için eğitilmiş AugCRNN-T ağırlıkları

## Settings
- Accelerator: **GPU T4**
- Internet: **ON**
- **Save & Run All (Commit)**

In [ ]:
# Hücre 1: ortam + repo
import torch, sys, os, subprocess, shutil, json
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}")
print(f"PyTorch {torch.__version__}")

REPO_URL = "https://github.com/Ridvan013/CRNN-Handwriting-Recognition.git"
BRANCH   = "feature/aachen-v3-extended-trigram"
REPO_DIR = "/kaggle/working/repo"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,REPO_DIR], check=True)
for name in ["cloud","aachen_splits","trigram_lm.py"]:
    src, dst = os.path.join(REPO_DIR,name), os.path.join("/kaggle/working",name)
    shutil.copytree(src,dst,dirs_exist_ok=True) if os.path.isdir(src) else shutil.copy(src,dst)
sys.path.insert(0,"/kaggle/working"); os.chdir("/kaggle/working")

import nltk
try: nltk.data.find("corpora/words")
except LookupError: nltk.download("words", quiet=True)
print("repo hazir")

In [ ]:
# Hücre 2: IAM veri yolu + eğitilmiş model yolu
import subprocess, os

res = subprocess.run(["find","/kaggle/input","-name","words.txt","-maxdepth","6"],
                     capture_output=True, text=True)
IAM_WORDS = IAM_ROOT = None
for wt in [p.strip() for p in res.stdout.splitlines() if p.strip()]:
    d = os.path.join(os.path.dirname(wt),"words")
    if os.path.isdir(d):
        IAM_WORDS, IAM_ROOT = wt, d; break
assert IAM_WORDS, "IAM dataset bulunamadi"
print(f"IAM words.txt : {IAM_WORDS}")
print(f"IAM words/    : {IAM_ROOT}")

res = subprocess.run(["find","/kaggle/input","-name","best_model_wa.pth"],
                     capture_output=True, text=True)
cands = [p.strip() for p in res.stdout.splitlines() if p.strip()]
TRAINED = max(cands, key=os.path.getsize) if cands else None
print(f"AugCRNN-T agirliklari: {TRAINED}")

---
## Tablo B — Lexicon / trigram ablation

Eğitim yok; tek modelin greedy çıktısına dört post-processing uygulanır (~15 dk).

In [ ]:
!python cloud/ablation_lexicon.py \
    --model {TRAINED} \
    --iam-words {IAM_WORDS} --iam-root {IAM_ROOT} \
    --out /kaggle/working/results/ablation_lexicon.json

---
## Tablo A — Augmentation ablation

Üç eğitim, her biri ~2 saat. Sadece `--aug-mode` değişir; diğer her şey sabittir.

In [ ]:
# 1/3 — wide photometric only (elastic ✗, morph ✗)
!python cloud/v3_augmented_train.py --aug-mode photo \
    --epochs 100 --batch 128 --lr 7e-4 --patience 15 \
    --model-dir /kaggle/working/abl_photo \
    --iam-words {IAM_WORDS} --iam-root {IAM_ROOT}

In [ ]:
# 2/3 — elastic only (morph ✗)
!python cloud/v3_augmented_train.py --aug-mode elastic \
    --epochs 100 --batch 128 --lr 7e-4 --patience 15 \
    --model-dir /kaggle/working/abl_elastic \
    --iam-words {IAM_WORDS} --iam-root {IAM_ROOT}

In [ ]:
# 3/3 — morphological only (elastic ✗)
!python cloud/v3_augmented_train.py --aug-mode morph \
    --epochs 100 --batch 128 --lr 7e-4 --patience 15 \
    --model-dir /kaggle/working/abl_morph \
    --iam-words {IAM_WORDS} --iam-root {IAM_ROOT}

In [ ]:
# Hücre 9: iki tabloyu da topla ve yazdır
import json, os, csv, math

def wa_from_csv(path):
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    k = sum(1 for r in rows if r.get("correct", r.get("Is_Correct","")).strip().lower() in ("1","true"))
    return k, len(rows), k/len(rows)*100

print("="*70); print(" TABLO A — Augmentation ablation"); print("="*70)
print(f"{'Configuration':<34s}{'WA (%)':>9s}{'CER (%)':>9s}")
print("-"*70)
print(f"{'CRNN-L (no elastic/morph)':<34s}{78.06:>9.2f}{11.16:>9.2f}   <- mevcut")
for tag, label in [("abl_photo","+ wide photometric"),
                   ("abl_elastic","+ elastic"),
                   ("abl_morph","+ morphological")]:
    j = f"/kaggle/working/results/v3_augmented_results.json"
    c = f"/kaggle/working/{tag}/test_results_analysis.csv"
    if os.path.exists(c):
        k,n,wa = wa_from_csv(c)
        print(f"{label:<34s}{wa:>9.2f}{'':>9s}   ({k}/{n})")
    else:
        print(f"{label:<34s}{'--':>9s}{'':>9s}   (kosulmadi)")
print(f"{'AugCRNN-T (elastic+morph)':<34s}{84.54:>9.2f}{9.21:>9.2f}   <- mevcut")

p = "/kaggle/working/results/ablation_lexicon.json"
if os.path.exists(p):
    r = json.load(open(p))
    print("\n"+"="*70); print(" TABLO B — Lexicon / trigram ablation"); print("="*70)
    print(f"{'Configuration':<42s}{'WA (%)':>9s}{'CER (%)':>9s}")
    print("-"*70)
    for c in r["configurations"]:
        print(f"{c['name']:<42s}{c['wa_pct']:>9.2f}{c['cer_pct']:>9.2f}")
    print(f"\nLexicon boyutlari: {r['lexicon_sizes']}")